EXTRA TREE REGRESSOR WITH HYPERTUNING

In [10]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [5]:
# LOAD DATA
df = pd.read_csv("/content/used_device_data.csv")

# FEATURE ENGINEERING
CURRENT_YEAR = 2026

df["device_age"] = CURRENT_YEAR - df["release_year"]

df["usage_ratio"] = (
    df["days_used"] /
    (df["device_age"] * 365 + 1)
)

df["camera_total"] = (
    df["rear_camera_mp"] +
    df["front_camera_mp"]
)

df["battery_per_weight"] = (
    df["battery"] /
    df["weight"]
)

df["ram_storage_ratio"] = (
    df["ram"] /
    df["internal_memory"]
)

In [7]:
# ENCODE CATEGORICALS
df["4g"] = df["4g"].map({
    "yes": 1,
    "no": 0
})

df["5g"] = df["5g"].map({
    "yes": 1,
    "no": 0
})

In [8]:
# TARGET
y = df["normalized_used_price"]
X = df.drop(
    columns=["normalized_used_price"]
)

X = pd.get_dummies(
    X,
    columns=["device_brand", "os"],
    drop_first=True
)

In [11]:
# SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [12]:
# EXTRA TREES MODEL
et = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1
)


In [13]:
# HYPERPARAMETER GRID
param_grid = {

    "n_estimators": [
        300,
        500,
        800,
        1000
    ],

    "max_depth": [
        10,
        15,
        20,
        25,
        None
    ],

    "min_samples_split": [
        2,
        5,
        10
    ],

    "min_samples_leaf": [
        1,
        2,
        4
    ],

    "max_features": [
        "sqrt",
        "log2",
        None
    ],

    "bootstrap": [
        True,
        False
    ]
}

In [14]:
# RANDOM SEARCH
search = RandomizedSearchCV(

    estimator=et,

    param_distributions=param_grid,

    n_iter=10,

    cv=3,

    scoring="r2",

    random_state=42,

    verbose=2,

    n_jobs=-1
)

search.fit(X_train, y_train)

print("\nBest Parameters:")
print(search.best_params_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best Parameters:
{'n_estimators': 1000, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 10, 'bootstrap': True}


In [15]:
# BEST MODEL
best_et = search.best_estimator_

In [16]:
# PREDICTIONS
y_pred = best_et.predict(X_test)

In [17]:
# EVALUATION
r2 = r2_score(y_test, y_pred)

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

print("\n===== EXTRATREES RESULTS =====")

print("R2 :", round(r2, 4))
print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))


===== EXTRATREES RESULTS =====
R2 : 0.8609
MAE: 0.1703
RMSE: 0.2124


In [18]:
best_et = search.best_estimator_

In [19]:
import joblib

joblib.dump(
    best_et,
    "extratrees_resale_model.pkl"
)

print("Model Saved Successfully!")

Model Saved Successfully!


In [20]:
import joblib

model = joblib.load(
    "extratrees_resale_model.pkl"
)

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [21]:
prediction = model.predict(X_test.iloc[[0]])

print(prediction)

[4.02366503]


In [22]:
joblib.dump(
    X.columns.tolist(),
    "model_columns.pkl"
)

['model_columns.pkl']